# Synthetic Data Validation for Model Selection

This notebook demonstrates the use of the `synth_validation` for evaluating
machine learning models using synthetic data with calibration.

## Overview

The framework implements:
1. **Synthetic Data Generation**: Using CTGAN, TVAE, GaussianCopula, TabPFGen, or TabDDPM
2. **Model Selection**: Training and evaluating 50+ model architectures
3. **Calibration**: Re-weighting synthetic samples to improve rank preservation
4. **Evaluation**: Spearman correlation and rank preservation metrics
5. **SHAP Analysis**: Interpreting calibration weights

## 1. Setup and Installation

In [ ]:
# Install the package in development mode
# !pip install -e .

# Or install dependencies manually:
# !pip install numpy pandas scipy scikit-learn sdv optuna matplotlib seaborn ucimlrepo

In [ ]:
# Add src to path for development
import sys
from pathlib import Path

# Navigate up from notebooks/ to project root, then into src/
src_path = str(Path().resolve().parent / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    
print(f"Added to path: {src_path}")

## 2. Import Package

In [ ]:
from synth_validation import (
    ExperimentRunner,
    DataLoader,
    SyntheticDataGenerator,
    setup_random_seeds,
    RANDOM_SEED
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
setup_random_seeds(RANDOM_SEED)

## 3. Available Datasets

The framework supports various UCI datasets for both classification and regression tasks.

In [ ]:
# Quick test - show just the first few datasets
loader = DataLoader()

print("Sample Classification Datasets:")
print("-" * 50)
for i, (name, info) in enumerate(loader.CLASSIFICATION_DATASETS.items()):
    if isinstance(info, dict):
        print(f"  - {name}: {info.get('description', 'No description')}")
        print(f"    Features: {info.get('features', 'Unknown')}, Samples: {info.get('samples', 'Unknown')}")
    else:
        print(f"  X {name}: {info} (ERROR: Expected dict, got {type(info)})")
print("...")

print("\nSample Regression Datasets:")
print("-" * 50)
for i, (name, info) in enumerate(loader.REGRESSION_DATASETS.items()):
    if isinstance(info, dict):
        print(f"  - {name}: {info.get('description', 'No description')}")
        print(f"    Features: {info.get('features', 'Unknown')}, Samples: {info.get('samples', 'Unknown')}")
    else:
        print(f"  X {name}: {info} (ERROR: Expected dict, got {type(info)})")
print("...")

## 4. Quick Start: Classification Example

Running a full K-fold calibration experiment on the Adult Income dataset.

In [ ]:
# Initialize experiment runner
runner = ExperimentRunner(
    dataset_name='diabetes',         # UCI Adult Income dataset
    synth_method='tabddpm',          # Use CTGAN for synthesis
    task_type='classification',      # Classification task
    loss_type='log_loss',            # Use log-loss for evaluation
    lambda_reg=0.5,                  # Regularization strength
    save_figures=True,               # Save figures to disk
    verbose=True                     # Print progress
)

In [ ]:
# Run K-fold calibration experiment
results = runner.run_kfold_calibration_experiment(
    n_folds=5,                       # Number of cross-validation folds
    M_calibration=10,                # Number of models for calibration
    synth_size_multiplier=1.0,       # Synthetic data size = test data size
    calib_test_ratio=0.2,            # 20% of train data for calibration test
    tune_synthetic=True,             # GAN tuning
    n_tune_trials=50,                # Number of tuning trials for synthetic data generator
    shap_plot_types=['bar', 'dot'],  # Types of SHAP plots to generate
    analyze_shap=True                # Perform SHAP analysis
)

## 5. Visualize Results

In [ ]:
# Correlation comparison (uncalibrated vs calibrated)
runner.visualize_correlation_results()

In [ ]:
# Loss distributions
runner.visualize_per_model_sample_losses(iteration=1)

## 6. Summary Statistics

In [ ]:
runner.compare_vectors(iteration=1)

In [ ]:
runner.print_summary_table()

## 7. Regression Example

In [ ]:
# Initialize for regression task
runner_reg = ExperimentRunner(
    dataset_name='california_housing',
    synth_method='tvae',
    task_type='regression',
    loss_type='mae',    # or 'mse'
    verbose=True
)

In [ ]:
# Run experiment
results_reg = runner_reg.run_kfold_calibration_experiment(
    n_folds=3,
    M_calibration=8,
    tune_synthetic=False,
    analyze_shap=False
)

In [ ]:
# Visualize regression results
runner_reg.visualize_correlation_results()

## 8. Using Different Synthesis Methods

The framework supports multiple synthetic data generation methods:

- `ctgan`: Conditional Tabular GAN (default)
- `tvae`: Tabular VAE
- `gaussian_copula`: Statistical copula-based method
- `tabpfgen`: Prior-Fitted Network with SGLD
- `tabddpm`: Denoising Diffusion Probabilistic Model

In [ ]:
# Compare synthesis methods
methods = ['ctgan', 'tvae', 'gaussian_copula', 'tabddpm', 'tabpfgen']
method_results = {}

for method in methods:
    print(f"\n{'='*60}")
    print(f"Testing method: {method}")
    print(f"{'='*60}")
    
    runner_temp = ExperimentRunner(
        dataset_name='adult',
        synth_method=method,
        task_type='classification',
        verbose=False
    )
    
    res = runner_temp.run_kfold_calibration_experiment(
        n_folds=3,
        M_calibration=8,
        analyze_shap=False
    )
    
    method_results[method] = {
        'uncalibrated': res['uncalibrated_stats']['mean'],
        'calibrated': res['calibrated_stats']['mean'],
        'improvement': res['calibrated_stats']['mean'] - res['uncalibrated_stats']['mean']
    }
    
    print(f"  Uncalibrated ρ: {method_results[method]['uncalibrated']:.4f}")
    print(f"  Calibrated ρ:   {method_results[method]['calibrated']:.4f}")
    print(f"  Improvement:    {method_results[method]['improvement']:+.4f}")

In [ ]:
# Visualize method comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(methods))
width = 0.35

uncalib_vals = [method_results[m]['uncalibrated'] for m in methods]
calib_vals = [method_results[m]['calibrated'] for m in methods]

ax.bar(x - width/2, uncalib_vals, width, label='Uncalibrated', color='coral', alpha=0.8)
ax.bar(x + width/2, calib_vals, width, label='Calibrated', color='lightgreen', alpha=0.8)

ax.set_xlabel('Synthesis Method')
ax.set_ylabel('Spearman Correlation')
ax.set_title('Comparison of Synthesis Methods')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Saving and Loading GAN Models

Trained GAN models can be saved and reused for faster experiments.

In [ ]:
# Save trained model
# model_path = runner.save_gan_model('adult_ctgan_experiment')
# print(f"Model saved to: {model_path}")

In [ ]:
# Load saved model
# runner_loaded = ExperimentRunner(
#     dataset_name='adult',
#     synth_method='ctgan',
#     task_type='classification'
# )
# runner_loaded.load_gan_model(model_path)

## 10. Advanced: Hyperparameter Tuning

Enable GAN hyperparameter tuning for better synthetic data quality.

In [ ]:
# # Run with hyperparameter tuning (takes longer)
# runner_tuned = ExperimentRunner(
#     dataset_name='adult',
#     synth_method='ctgan',
#     task_type='classification',
#     verbose=True
# )
# 
# results_tuned = runner_tuned.run_kfold_calibration_experiment(
#     n_folds=5,
#     M_calibration=10,
#     tune_synthetic=True,      # Enable tuning
#     n_tune_trials=20,         # Number of Optuna trials
#     analyze_shap=True
# )